The point of this notebook is to:
- determine the length of time it will take to use libraries to process text detection.
- test optimisation strategies.

Aims/requirements:
- All profanity should be filtered out, particularly those in subtitles. Starred words should also be removed.
- Profanity filtering should only take ~1 second per second of footage. This ensures an hour video can be filtered in ~1 hour
- Ideally, this process should take < 0.5 seconds per second of footage.

In [1]:
## Imports and global constants
import cv2
import easyocr
import time

video_path = "../data/raw/example1.mp4"

Load the profanity file

In [2]:
# Load profanity
profanity_set = set()
profanity_file_path = "../src/content_filter/config/profanity_words.txt"
with open(profanity_file_path, "r", encoding="utf-8") as f:
    for line in f:
        word = line.strip().lower()
        profanity_set.add(word)

def contains_profanity(text):
    words = text.lower().split()
    return any(word.strip(".,!?") in profanity_set for word in words)

Define a function for benchmarking EasyOCR

In [3]:
def run_easy_ocr_test(duration):
    reader = easyocr.Reader(['en'], gpu=False, quantize=True)
    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"FPS: {fps}")

    frame_count = 0

    profanity_boxes = []

    max_frames = int(fps * duration)  # first 3 seconds
    t0 = time.time()
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        # Run OCR on frame
        results = reader.readtext(frame)

        for (bbox, text, confidence) in results:
            if contains_profanity(text):
                # Convert bbox points to int
                bbox = [(int(x), int(y)) for (x, y) in bbox]
                profanity_boxes.append((frame_count, bbox))

                # Draw rectangle on frame
                top_left = bbox[0]
                bottom_right = bbox[2]
                cv2.rectangle(frame, top_left, bottom_right, (0, 0, 255), 2)

        frame_count += 1

    t1 = time.time()
    cap.release()
    cv2.destroyAllWindows()
    return t1 - t0


In [4]:
# duration = 3
# dt = run_easy_ocr_test(duration)
# print(f"It took: {dt} seconds to process all of the footage")
# print(f"Time taken per second of footage: {(dt)/duration :.2f}s")

#### Findings for naive method using EasyOCR on CPU:
- Time taken per second of footage: 181.64s
- 3 minutes per second of footage is WAY too much time.
- Cannot use GPU as the project should run locally on a machine.

In [10]:
### Trying Tesseract ###
import pytesseract

def run_tesseract_ocr(seconds, output_path):
    cap = cv2.VideoCapture(video_path)

    frame_count = 0
    profanity_boxes = []

    fps = cap.get(cv2.CAP_PROP_FPS)
    max_frames = int(fps * seconds)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    t0 = time.time()
    while frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        # Get detailed OCR data (word-level boxes)
        data = pytesseract.image_to_data(frame, output_type=pytesseract.Output.DICT)

        n_boxes = len(data["text"])

        fourcc = cv2.VideoWriter.fourcc('m', 'p', '4', 'v')  # use 'XVID' if mp4v fails
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        for i in range(n_boxes):
            word = data["text"][i]
            conf = int(data["conf"][i])

            if conf > 60 and contains_profanity(word):
                x = data["left"][i]
                y = data["top"][i]
                w = data["width"][i]
                h = data["height"][i]

                profanity_boxes.append((frame_count, (x, y, w, h)))

                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)

        frame_count += 1
        out.write(frame)

    t1 = time.time()
    cap.release()
    cv2.destroyAllWindows()

    return (t1-t0), n_boxes


In [16]:
dt, _ = run_tesseract_ocr(3, "../data/processed/censored_text.mp4")

In [15]:
print(f"It took: {dt}")
print(f"Time taken per second of footage: {(dt)/3 :.2f}s")

It took: 28.711276054382324
Time taken per second of footage: 9.57s


#### Tesseract naive findings
- Time taken per second of footage: 8.85s on average
- Significantly faster, and what I will be using for this project as easyOCR is far too heavy.
- Still not fast enough: a 1 hour video will take 9 hours to filter, which is still too long.

##### Testing quality of approaches: naive
In theory, the naive approach should be the most accurate as we are running the OCR every frame.

In [ ]:
dt, base_text_boxes = run_tesseract_ocr()